<a href="https://colab.research.google.com/github/JeroCQ/Meeting-Summarizer/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Assumptions & Baseline Metrics
- Meeting Volume: 20 leadership meetings per month.
- Average Duration: 60 minutes per meeting.
- Total Monthly Audio: 1,200 minutes.
- Average Transcript Length: 9,000 words (12,000 tokens) per meeting.

#Fixed & Infrastructure Costs
- Cloud Storage (GCP Cloud Storage): s1.00 / month (For temporarily storing raw audio files and processed JSON logs).
- Orchestration (GCP Cloud Run): s0.00 (Processing  just 20 executions per month is free).
- Google Slides API Usage: s0.00 (Standard usage limits far exceed 20 presentations per month).
- Subtotal Fixed Costs: s1.00 / month.

#Variable Costs (Scales with Volume)
- Speech-to-Text Transcription: s0.006 per minute × 1,200 minutes = s7.20 / month.
- AI Inference (Gemini 1.5 Flash):
- Input Tokens: 240,000 tokens / month at s0.075 per 1M = s0.02
- Output Tokens: 10,000 tokens / month at s0.30 per 1M = s0.003

Note: Gemini Flash is selected for its highly optimized long-context processing, keeping inference costs near zero.

- Subtotal Variable Costs: s7.22 / month.
- Total Estimated Monthly Cost: s8.22

#Cost Analysis & Scaling
At s0.41 per generated presentation, this serverless architecture is extremely lightweight and scales linearly. The primary cost driver is the Speech-to-Text transcription. If the meeting volume scales to 100 meetings per month, the total infrastructure cost would still remain under $45.00 monthly.

In [ ]:
import os
from google.colab import auth
from googleapiclient.discovery import build
import google.generativeai as genai
from pydantic import BaseModel, Field
from typing import List
from google.colab import userdata

# Getting myself logged into Drive and Slides so the script can actually move files around
auth.authenticate_user()

# Setting up the workspace services - I'll use these to copy the template and fill it in
drive_service = build('drive', 'v3')
slides_service = build('slides', 'v1')

# Grab my Gemini API key from my Colab secrets (it's way safer than hardcoding it here)
GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# The ID for my slide template - this is the 'master' deck we're copying from
TEMPLATE_PRESENTATION_ID = "1R1dCAszs4HqjpvGMa7CIsK7cul79uvVJ6n2qnnnM-3Q"

In [ ]:
# Just a quick check to see which models I actually have access to right now
# Sometimes naming conventions change so I like to list them out to be sure
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
mod

In [ ]:
# Here I'm defining the exact structure I want my meeting summary to follow
# Using Pydantic is great because it forces the AI to give me data in a format I can actually use later
class MeetingSummary(BaseModel):
    executive_summary: str = Field(description="High-level executive summary")
    objectives: List[str] = Field(description="Exactly 3 high-level objectives")
    actionable_items: List[str] = Field(description="Exactly 3 actionable items")
    next_steps: str = Field(description="Clear next steps for the team")

In [ ]:
# I'm skipping the complex audio processing for now and just loading my transcript text directly
# This makes it way faster to test the summary logic without waiting for a file to upload
with open("synthetic_retail_ai_transcript.txt", "r") as file:
    transcript_text = file.read()

print("Transcript loaded successfully. Total characters:", len(transcript_text))

Transcript loaded successfully. Total characters: 20797


In [ ]:
# This is where I'm calling the AI. I'm using gemini-flash because it's fast and cheap for long texts
model = genai.GenerativeModel('models/gemini-flash-latest')

# Converting my Pydantic class to a JSON schema so the AI knows the exact rules
schema_json = MeetingSummary.model_json_schema()
prompt = f"""Task: Summarize the transcript into a JSON object.
Constraint: Return ONLY valid JSON. Do not include markdown formatting or extra text.
Schema: {schema_json}
Transcript: {transcript_text}"""

try:
    # I'm setting max_output_tokens high and temperature low to keep the summary detailed but consistent
    response = model.generate_content(
        prompt,
        generation_config={
            "response_mime_type": "application/json",
            "max_output_tokens": 4096,
            "temperature": 0.1
        },
        # Disabling safety blocks here just so normal business content doesn't get randomly flagged
        safety_settings=[
            {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
            {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}
        ]
    )

    # Cleaning up the text and feeding it back into Pydantic to make sure everything lines up
    json_text = response.text.strip()
    summary_data = MeetingSummary.model_validate_json(json_text)
    print("Success: Summary generated and parsed.")
except Exception as e:
    # If it fails, I want to see exactly what happened and how much text I got back to debug
    print(f"Failed with error: {e}")
    if 'response' in locals():
        print("\nRaw response length:", len(response.text))
        print("Raw response head:", response.text[:100])
        print("Raw response tail:", response.text[-100:])

Failed with error: 1 validation error for MeetingSummary
  Invalid JSON: EOF while parsing an object at line 13 column 248 [type=json_invalid, input_value='{\n  "executive_summary"...rsus scalable options."', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid

Raw response length: 1677
Raw response head: {
  "executive_summary": "BrightLane Retail is seeking a practical, cost-effective AI automation sol
Raw response tail: sk assessment of potential failure points, and a comparison of lightweight versus scalable options."


In [ ]:
model = genai.GenerativeModel('gemini-pro-latest')

# Send the Pydantic Scheme to gemini
prompt = f"Analyze this transcript. Return ONLY a valid JSON object matching this exact schema: {MeetingSummary.model_json_schema()}. Transcript: {transcript_text}"

# Send the content generation to gemini
response = model.generate_content(prompt)

# Markdown cleansing
clean_json = response.text.replace('```json', '').replace('```', '').strip()

# check data with MeetingSummary
summary_data = MeetingSummary.model_validate_json(clean_json)
print("Listo! Ya tenemos el resumen parseado con Pro.")

Listo! Ya tenemos el resumen parseado con Pro.


In [ ]:
# I'm making a fresh copy of my template deck so I don't overwrite my master file by mistake
# This gives me a brand new presentation ID to work with for this specific meeting output
copied_file = drive_service.files().copy(
    fileId=TEMPLATE_PRESENTATION_ID,
    body={'name': 'Generated Meeting Presentation'}
).execute()
new_deck_id = copied_file.get('id')
print(f"New presentation created with ID: {new_deck_id}")

New presentation created with ID: 14wQzLHlDDXEqmAaRlQeXRZ8I6K3v01T-p5ggXPdAXfk


In [ ]:
# Now for the fun part: swapping out my placeholder tags like {{EXECUTIVE_SUMMARY}} with the real data
# I'm building a list of 'requests' for the Google Slides API to process
requests = [
    {
        'replaceAllText': {
            'containsText': {'text': '{{EXECUTIVE_SUMMARY}}', 'matchCase': True},
            'replaceText': summary_data.executive_summary
        }
    },
    {
        'replaceAllText': {
            'containsText': {'text': '{{NEXT_STEPS}}', 'matchCase': True},
            'replaceText': summary_data.next_steps
        }
    }
]

# I'll loop through the first 3 objectives and action items to fill those specific slots in the slides
for i in range(3):
    obj_text = summary_data.objectives[i] if i < len(summary_data.objectives) else ""
    act_text = summary_data.actionable_items[i] if i < len(summary_data.actionable_items) else ""

    requests.append({
        'replaceAllText': {
            'containsText': {'text': f'{{{{OBJ_{i+1}}}}}', 'matchCase': True},
            'replaceText': obj_text
        }
    })
    requests.append({
        'replaceAllText': {
            'containsText': {'text': f'{{{{ACT_{i+1}}}}}', 'matchCase': True},
            'replaceText': act_text
        }
    })

# Sending the whole batch of changes at once to the Slides API - it's much faster than doing one by one
slides_service.presentations().batchUpdate(
    presentationId=new_deck_id,
    body={'requests': requests}
).execute()

print(f"Success! View your slides at: https://docs.google.com/presentation/d/{new_deck_id}/edit")

Success! View your slides at: https://docs.google.com/presentation/d/14wQzLHlDDXEqmAaRlQeXRZ8I6K3v01T-p5ggXPdAXfk/edit
